# 01 — Exploratory Data Analysis

This notebook explores the synthetic banking dataset before the ETL pipeline publishes analytical outputs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path('../data/raw')
customers = pd.read_csv(RAW/'customers.csv', parse_dates=['customer_since'])
accounts = pd.read_csv(RAW/'accounts.csv', parse_dates=['opening_date'])
transactions = pd.read_csv(RAW/'transactions.csv', parse_dates=['transaction_date'])
products = pd.read_csv(RAW/'products.csv')
customer_products = pd.read_csv(RAW/'customer_products.csv', parse_dates=['start_date'])

print(customers.shape, accounts.shape, transactions.shape, products.shape, customer_products.shape)

## Customer profile

In [ ]:
customers[['age','annual_income']].describe(percentiles=[.5,.9,.99]).round(2)

In [ ]:
customers['city'].value_counts().plot(kind='bar', figsize=(10,4), title='Customers by City')
plt.tight_layout()

In [ ]:
customers['occupation'].value_counts().plot(kind='bar', figsize=(10,4), title='Customers by Occupation')
plt.tight_layout()

## Accounts

In [ ]:
accounts[['balance']].describe(percentiles=[.5,.9,.99]).round(2)

In [ ]:
accounts['account_type'].value_counts().plot(kind='bar', title='Account Type Mix')
plt.tight_layout()

## Transactions

In [ ]:
transactions[['amount']].describe(percentiles=[.5,.9,.99]).round(2)

In [ ]:
transactions['channel'].value_counts().plot(kind='bar', title='Transaction Channels')
plt.tight_layout()

In [ ]:
transactions['year_month'] = transactions['transaction_date'].dt.to_period('M').astype(str)
monthly = transactions[transactions.status=='Success'].groupby('year_month').agg(transactions=('transaction_id','count'), transaction_value=('amount','sum')).reset_index()
monthly.head()

## Relationship checks

In [ ]:
account_counts = accounts.groupby('customer_id').size()
transaction_counts = transactions.groupby('customer_id').size()
pd.DataFrame({
    'customers_with_1_account_pct': [(account_counts==1).mean()*100],
    'customers_with_2plus_accounts_pct': [(account_counts>=2).mean()*100],
    'customers_with_0_transactions': [(~customers.customer_id.isin(transaction_counts.index)).sum()]
}).round(2)